----
# **<span style="color:DarkSlateBlue">PROYECTO BOOKING_REVIEWS</span>**
# **<span style="color:DarkSlateBlue">Limpieza de datos</span>**
----

---
---
## <span style="color:gray">**1. Importación de librerías**</span> 📂

In [55]:
# Tratamiento de datos
import numpy as np
import pandas as pd 
from IPython.display import display
import pycountry

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de ruta
import sys
sys.path.append('../')

# Importación de funciones personalizadas
from src.soporte2_limpieza import *


---
---
## <span style="color:gray">**2. Carga de datos**</span> 📥

In [56]:
booking_raw = pd.read_csv("../data/raw/hotel_booking.csv")

reviews_raw = pd.read_csv("../data/raw/hotel_reviews.csv")

In [57]:
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

---
---

----
# <span style="color:DarkSlateBlue">**Desarrollo del proyecto - 2**</span>
----


---
---
## <span style="color:gray">**Limpieza de datos**</span> 🧹

Antes de continuar con el análisis exploratorio de datos (EDA), creamos copias de los DataFrames originales.
Esto nos permite manipular, limpiar o transformar los datos sin afectar los originales, asegurando que siempre podamos recuperar la información original si es necesario.

In [58]:
booking_limpio = booking_raw.copy()
reviews_limpio = reviews_raw.copy()

---
## `booking_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>


La única columna que habría que estandarizar es `phone-number`, ya que utiliza como separador "-", en lugar de "_" como el resto de nombres de columnas. Sin embargo, al ser irrelevante para nuestro análisis, la eliminaremos en el próximo paso.

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `arrival_date_week_number` y `arrival_date_day_of_month`

Solo necesitamos el mes y año de llegada para poder unir este dataset con el de reviews. Por lo tanto, estas columnas se pueden eliminar.

- `agent`y `company`

Representan los ID de las agencias y compañías. No aportan información relevante para nuestro análisis, por lo que se eliminan.

- `days_in_waiting_list`

La mayoría de los huéspedes tienen 0 días en espera y no es una información útil para el análisis que queremos realizar. En consecuncia, la eliminaremos.

- `required_car_parking_spaces` y `total_of_special_requests`

El valor promedio es muy bajo, la desviación estándar es baja y los percentiles muestran que la mayoría de los huéspedes no requiere estas opciones. Estas columnas aportan poca información útil y se eliminan.

- `name`, `email`, `phone-number`, `credit_card` 

Nos dan información sensible del huésped que no es que no aporta valor al análisis. Se eliminan.

**Eliminamos las columnas anteriores**

In [59]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
booking_limpio = eliminar_columns(
    booking_limpio,
    ["arrival_date_week_number", "arrival_date_day_of_month", 
     "agent", "company", "days_in_waiting_list", 
     "required_car_parking_spaces", "total_of_special_requests",
     "name", "email", "phone-number", "credit_card"]
)

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos la abreviatura de los países por el nombre completo del país.**

In [60]:
def code_to_name(code):
    try:
        return pycountry.countries.get(alpha_3=code.upper()).name
    except:
        return None

booking_limpio['country'] = booking_limpio['country'].apply(code_to_name)

- **Comprobamos los datos únicos de las variables categóricas y hacemos la limpieza de algunos de los datos.**

In [61]:
col_cat = booking_limpio.select_dtypes(include=['category', 'str']).columns
datos_unicos(booking_limpio, col_cat)



Los datos únicos de la varible hotel son:

 <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible arrival_date_month son:

 <StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible meal son:

 <StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str


-----------------------------------------------------------------


Los datos únicos de la varible country son:

 <StringArray>
[                        'Portugal',                   'United Kingdom',
                    'United States',                            'Spain',
                          'Ireland',                           'France',
                             

- Todas las variables presentan mayúsculas y minúsculas, lo estandarizamos todo a minúsculas.
- Las variables `hotel`, `deposit_type` y `market_segment` tienen algunos datos con espacio entre medias, lo cambiamos por "_".
- `market_segment` y `distribution_channel` tienen el caracter "/", el cual sustituimos por "_".
- `customer_type` y `reservation_status` tienen como separador de palabras "-", lo cambiamos por "_".


In [62]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
minus(booking_limpio, col_cat)

# Llamamos a la función reemplazar guardada en el archivo de soporte 
reemplazar(booking_limpio, ["hotel", "country", "deposit_type", "market_segment"], " ", "_")
reemplazar(booking_limpio, ["market_segment", "distribution_channel"], "/", "_")
reemplazar(booking_limpio, ["customer_type", "reservation_status"], "-", "_")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Cambiamos las variables `is_canceled` y `is_repeated_guest` a variables categóricas:** 

   - 1 = 'si'
   - 0 = 'no'

In [63]:
columnas = ["is_canceled", "is_repeated_guest"]

for columna in columnas:
    booking_limpio[columna] = booking_limpio[columna].replace({1:'si', 0:'no'})

- **Variable `children`**

Para convertir esta variable a números enteros, primero hay que pasar de string a float, hacer un redondeo al entero más cercano y finalmente se cambia a int:

In [64]:
booking_limpio['children'] = booking_limpio['children'].astype('Int64')

- **Variable `reservation_status_date`**

In [65]:
booking_limpio["reservation_status_date"] = (
    pd.to_datetime(booking_limpio["reservation_status_date"], 
                   format="%Y-%M-%d", 
                   errors="coerce")
                   )

Una vez convertida la variable a formato fecha, se crearán dos nuevas variables: una que contenga el año y otra el mes, con el objetivo de obtener información temporal más relevante para el análisis.

In [66]:
booking_limpio['reservation_status_year'] = booking_limpio['reservation_status_date'].dt.year

In [67]:
booking_limpio['reservation_status_month'] = booking_limpio['reservation_status_date'].dt.month_name()

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

Durante la exploración inicial del Dataset, identificamos columnas con valores nulos; a continuación, evaluaremos cómo tratarlos individualmente.

Antes de tratar los nulos generamos un DataFrame que contenga solo las columnas que tienen nulos:

In [68]:
df_nulos = booking_limpio.loc[:, booking_limpio.isnull().sum() > 0]
df_nulos.columns

Index(['children', 'country'], dtype='str')

Recordamos el porcentaje de nulos de cada columna:

In [69]:
(df_nulos.isnull().sum() / (df_nulos.shape[0])*100).round(3)

children    0.003
country     1.483
dtype: float64

- Variable `children`

Lo imputamos con la moda.

In [70]:
booking_limpio["children"] = booking_limpio["children"].fillna(booking_limpio["children"].mode()[0])


- Variable `country`

Sustituímos los valores nulos por 'unknown'.

In [71]:
booking_limpio["country"] = booking_limpio["country"].fillna('unknown') 

---
## `reviews_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>

- Estandarización de nombres de las columnas

In [72]:
estandar_columns(reviews_limpio)

### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- `review_total_negative_word_counts` y `review_total_positive_word_counts` 
- `lat` y `lng`

**Eliminamos las columnas anteriores**

In [73]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
reviews_limpio = eliminar_columns(reviews_limpio, ["review_total_negative_word_counts", "review_total_positive_word_counts", "lat", "lng"])

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Cambiamos los datos de las variables categóricas a minúscula**

In [74]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
col_cat = booking_limpio.select_dtypes(include=['category']).columns
minus(reviews_limpio, col_cat)

### <span style="color:darkgray">**6. Eliminación de duplicados**</span>

In [ ]:
reviews_limpio = reviews_limpio.drop_duplicates()

---
---
## <span style="color:gray">**Validación final de los datasets**</span> ✅

- **Comprobamos que la limpieza de ambos dataset se ha realizado correctamente**


### **`booking_limpio`** 

In [ ]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(booking_limpio)

Y los datos de la variable `reservation_status_month` empiezan por mayúsculas, los cambiamos a minúsculas para mantener la consistencia en todo el DataFrame:

In [ ]:
minus(booking_limpio, ['reservation_status_month'])

### **`reviews_limpio`** 

In [ ]:
# Llamamos a la función analisis_rapido guardada en el archivo de soporte
analisis_rapido(reviews_limpio)

Hemos verificado que la limpieza de ambos dataframes se ha realizado correctamente: 

- Los nombres de las columnas son claros y descriptivos.
- Los datos son consistentes entre sí.
- No se observan valores nulos.
- Las variables presentan formatos adecuados para análisis y modelado posteriores .

Además, las transformaciones aplicadas (como estandarización de texto, conversión de tipos numéricos y codificación de variables categóricas) aseguran que ambos dataframes puedan combinarse y utilizarse de manera confiable.

- **Guardamos los dataset limpios**

In [ ]:
booking_limpio.to_csv("../data/output/booking_limpio.csv", index=False)
reviews_limpio.to_csv("../data/output/reviews_limpio.csv", index=False)

---